# Meta-NATH VisA Full Demo Kaggle Workflow

Thin Kaggle orchestration notebook for the separate VisA benchmark. It clones or updates the repo, prepares the VisA dataset from Kaggle inputs or an optional URL/archive, runs the conservative Phase 3 path, mechanism smoke, the experimental or max-power Phase 3 path, prints reports, and packages artifacts.

## Before Running

- Kaggle accelerator: GPU.
- Internet: on, unless the repo/model cache and VisA archive are mounted as Kaggle inputs.
- Add a Kaggle input that contains either extracted VisA category folders or a VisA archive. The expected category folders include `candle`, `capsules`, `cashew`, ...
- If your VisA source is a direct URL, set `VISA_URL` in the first cell or as a Kaggle environment variable.
- Set `MAX_POWER=1` for the VisA max-power experimental profile.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from tqdm.auto import tqdm

REPO_URL = "https://github.com/taitrn/NestedLearningForCAD.git"
BRANCH = "taitrn"
REPO = Path("/kaggle/working/NestedLearningForCAD")

# Change these if your Kaggle input names differ.
VISA_INPUT = Path(os.environ.get("VISA_INPUT", "/kaggle/input/visa"))
VISA_ARCHIVE = os.environ.get("VISA_ARCHIVE", "")
VISA_URL = os.environ.get("VISA_URL", "")

# Empty MAX_TASKS means all VisA classes. Set MAX_TASKS=3 for a quick smoke run.
MAX_TASKS = os.environ.get("MAX_TASKS", "")
RUN_PHASE3 = os.environ.get("RUN_PHASE3", "1")
RUN_MECHANISM = os.environ.get("RUN_MECHANISM", "1")
RUN_EXPERIMENTAL = os.environ.get("RUN_EXPERIMENTAL", "1")
REQUIRE_ACCEPTED = os.environ.get("REQUIRE_ACCEPTED", "0")
REQUIRE_EXPERIMENTAL_ACCEPTED = os.environ.get("REQUIRE_EXPERIMENTAL_ACCEPTED", "0")
MAX_POWER = os.environ.get("MAX_POWER", "0")
STEP_TIMEOUT_SECONDS = os.environ.get("STEP_TIMEOUT_SECONDS", "7200")
METANATH_REQUIRE_HF_BACKBONE = os.environ.get("METANATH_REQUIRE_HF_BACKBONE", "1")

CONFIG = os.environ.get("CONFIG", "conf/visa_phase3.yaml")
DEFAULT_EXPERIMENTAL_CONFIG = "conf/visa_max_power.yaml" if MAX_POWER == "1" else "conf/visa_experimental_nsp2_cbp.yaml"
EXPERIMENTAL_CONFIG = os.environ.get("EXPERIMENTAL_CONFIG", DEFAULT_EXPERIMENTAL_CONFIG)
SCRIPT = "scripts/run_server_visa.sh"
INSTALL_DEPS = False
INCLUDE_CHECKPOINTS_IN_ZIP = False

def run(cmd, cwd=None, env=None, heartbeat=60):
    cmd = [str(x) for x in cmd]
    start = time.time()
    next_beat = heartbeat
    print(f"\n[{time.strftime('%H:%M:%S')}] $ {' '.join(cmd)}", flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, env=env)
    while proc.poll() is None:
        elapsed = int(time.time() - start)
        if heartbeat and elapsed >= next_beat:
            print(f"[{time.strftime('%H:%M:%S')}] still running ({elapsed}s): {' '.join(cmd[:3])}", flush=True)
            next_beat += heartbeat
        time.sleep(2)
    elapsed = int(time.time() - start)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    print(f"[{time.strftime('%H:%M:%S')}] done ({elapsed}s)", flush=True)

print("repo:", REPO)
print("branch:", BRANCH)
print("visa_input:", VISA_INPUT)
print("visa_archive:", VISA_ARCHIVE or "<auto>")
print("visa_url:", VISA_URL or "<not set>")
print("config:", CONFIG)
print("script:", SCRIPT)
print("max_tasks:", MAX_TASKS or "all")
print("run_phase3:", RUN_PHASE3)
print("run_mechanism:", RUN_MECHANISM)
print("run_experimental:", RUN_EXPERIMENTAL)
print("max_power:", MAX_POWER)
print("experimental_config:", EXPERIMENTAL_CONFIG)
print("step_timeout_seconds:", STEP_TIMEOUT_SECONDS)
print("metanath_require_hf_backbone:", METANATH_REQUIRE_HF_BACKBONE)


## Clone Or Update Repo

In [ ]:
if REPO.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=REPO)
    run(["git", "checkout", BRANCH], cwd=REPO)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO)
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO])

print("Repo ready:", REPO)


## Install Dependencies And Prepare VisA Dataset

In [ ]:
if INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO)
else:
    print("Skipping pip install. Set INSTALL_DEPS=True if Kaggle image lacks dependencies.")

import tarfile
import urllib.error
import urllib.request

VISA_CATS = [
    "candle", "capsules", "cashew", "chewinggum", "fryum", "macaroni1",
    "macaroni2", "pcb1", "pcb2", "pcb3", "pcb4", "pipe_fryum",
]
ARCHIVE_SUFFIXES = (".zip", ".tar", ".tar.gz", ".tgz", ".tar.xz", ".txz")

def has_visa_categories(path):
    if not path.exists():
        return False
    hits = 0
    for cat in VISA_CATS[:4]:
        cat_dir = path / cat
        if cat_dir.is_dir():
            hits += 1
    return hits >= 3

def looks_like_visa_archive(path):
    name = path.name.lower()
    return path.is_file() and name.endswith(ARCHIVE_SUFFIXES) and ("visa" in name or "visual_anomaly" in name)

def candidate_dataset_roots(preferred):
    candidates = []
    if preferred.exists():
        candidates.append(preferred)
        candidates.extend([p for p in preferred.iterdir() if p.is_dir()])
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for ds in input_root.iterdir():
            if ds.is_dir():
                candidates.append(ds)
                candidates.extend([p for p in ds.iterdir() if p.is_dir()])
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        yield candidate

def find_dataset_root(preferred):
    for candidate in candidate_dataset_roots(preferred):
        if has_visa_categories(candidate):
            return candidate
    raise FileNotFoundError("Could not find extracted VisA category folders under /kaggle/input.")

def find_visa_archive():
    if VISA_ARCHIVE:
        archive = Path(VISA_ARCHIVE)
        if not archive.exists():
            raise FileNotFoundError(f"VISA_ARCHIVE does not exist: {archive}")
        return archive
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    matches = [p for p in input_root.rglob("*") if looks_like_visa_archive(p)]
    return matches[0] if matches else None

def check_safe_extract_path(target_root, member_name):
    root = target_root.resolve()
    member_path = (target_root / member_name).resolve()
    if root != member_path and root not in member_path.parents:
        raise RuntimeError(f"Unsafe path in archive: {member_name}")

def safe_extract_archive(archive, extract_to):
    extract_to.mkdir(parents=True, exist_ok=True)
    archive = Path(archive)
    name = archive.name.lower()
    print("Extracting VisA archive:", archive)
    print("Destination:", extract_to)
    if name.endswith(".zip"):
        with zipfile.ZipFile(archive) as zf:
            members = zf.infolist()
            for member in members:
                check_safe_extract_path(extract_to, member.filename)
            for member in tqdm(members, desc="extract visa zip", unit="file"):
                zf.extract(member, extract_to)
    elif name.endswith(ARCHIVE_SUFFIXES):
        mode = "r:*"
        with tarfile.open(archive, mode) as tf:
            members = tf.getmembers()
            for member in members:
                check_safe_extract_path(extract_to, member.name)
            for member in tqdm(members, desc="extract visa tar", unit="file"):
                tf.extract(member, extract_to)
    else:
        raise ValueError(f"Unsupported VisA archive: {archive}")

def download_visa_archive(target):
    if not VISA_URL:
        return None
    target.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading VisA from:", VISA_URL)
    request = urllib.request.Request(VISA_URL, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response, target.open("wb") as out:
            total = int(response.info().get("Content-Length", 0) or 0)
            done = 0
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)
                done += len(chunk)
                if total:
                    print(f"  {done / (1024 ** 2):.1f}/{total / (1024 ** 2):.1f} MB", end="\r")
        print("\nDownloaded:", target)
        return target
    except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, OSError) as exc:
        if target.exists():
            target.unlink()
        raise RuntimeError(f"VisA download failed: {exc}") from exc

def prepare_visa_dataset():
    try:
        return find_dataset_root(VISA_INPUT)
    except FileNotFoundError:
        pass

    extract_to = Path("/kaggle/working/visa_extracted")
    if has_visa_categories(extract_to):
        return find_dataset_root(extract_to)

    archive = find_visa_archive()
    if archive is None:
        suffix = ".zip"
        if VISA_URL.lower().endswith((".tar", ".tar.gz", ".tgz", ".tar.xz", ".txz")):
            suffix = ".tar"
        archive = download_visa_archive(Path("/kaggle/working/visa_dataset" + suffix))
    if archive is None:
        raise FileNotFoundError(
            "Could not find VisA data. Add a Kaggle input with extracted VisA folders or a VisA archive, "
            "or set VISA_URL/VISA_ARCHIVE in the first cell."
        )
    safe_extract_archive(archive, extract_to)
    return find_dataset_root(extract_to)

def link_dataset(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        if has_visa_categories(dst):
            print("Dataset target already exists:", dst)
            return
        raise FileExistsError(f"Dataset target exists but does not look like VisA: {dst}")
    try:
        os.symlink(src, dst, target_is_directory=True)
        print("Symlinked", src, "->", dst)
    except OSError:
        print("Symlink failed; copying VisA dataset. This may take a while.")
        shutil.copytree(src, dst)

visa_root = prepare_visa_dataset()
link_dataset(visa_root, REPO / "data" / "visa")
print("VisA root:", visa_root)


## Sanity Check

In [ ]:
print("Python:", sys.executable)
run([sys.executable, "-m", "py_compile", "conf/config.py", "dataset/load_dataset.py"], cwd=REPO)
run([sys.executable, "-m", "py_compile", "training/run_experiment.py", "training/consolidation_engine.py", "training/meta_nath_engine.py"], cwd=REPO)
run([sys.executable, "-m", "py_compile", "scripts/pipeline/run_phase3_consolidation.py", "scripts/pipeline/evaluate_checkpoint.py", "scripts/pipeline/phase3_acceptance.py"], cwd=REPO)
run(["bash", "-n", SCRIPT, "scripts/workflows/run_server_visa.sh"], cwd=REPO)
assert (REPO / CONFIG).exists(), f"Config not found: {CONFIG}"
assert (REPO / SCRIPT).exists(), f"Script not found: {SCRIPT}"
assert (REPO / "data" / "visa").exists(), "VisA link was not created"
print("Preflight OK.", flush=True)


## Run VisA Full Demo Workflow

This calls `scripts/run_server_visa.sh`. By default it runs the conservative VisA Phase 3 path, mechanism smoke, and the experimental NSP2/CBP path. Set `MAX_POWER=1` or `EXPERIMENTAL_CONFIG=conf/visa_max_power.yaml` for the max-power experimental profile. Set `RUN_PHASE3=0 RUN_EXPERIMENTAL=0` only for a Phase 1-2 smoke run.

In [ ]:
env = os.environ.copy()
env.update({
    "PYTHON_BIN": sys.executable,
    "CONFIG": CONFIG,
    "EXPERIMENTAL_CONFIG": EXPERIMENTAL_CONFIG,
    "RUN_PHASE3": RUN_PHASE3,
    "RUN_MECHANISM": RUN_MECHANISM,
    "RUN_EXPERIMENTAL": RUN_EXPERIMENTAL,
    "REQUIRE_ACCEPTED": REQUIRE_ACCEPTED,
    "REQUIRE_EXPERIMENTAL_ACCEPTED": REQUIRE_EXPERIMENTAL_ACCEPTED,
    "PROGRESS": "1",
    "STEP_TIMEOUT_SECONDS": STEP_TIMEOUT_SECONDS,
    "HF_HUB_DISABLE_XET": "1",
    "METANATH_REQUIRE_HF_BACKBONE": METANATH_REQUIRE_HF_BACKBONE,
})
if MAX_TASKS:
    env["MAX_TASKS"] = str(MAX_TASKS)

run(["bash", SCRIPT], cwd=REPO, env=env)


## Inspect VisA Full Demo Manifest

In [ ]:
def latest_file(pattern):
    print(f"Searching: {pattern}", flush=True)
    files = []
    for path in tqdm(REPO.glob(pattern), desc="matching files", unit="file"):
        files.append(path)
    files = sorted(files, key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(pattern)
    print(f"Found {len(files)} match(es); using latest: {files[-1]}", flush=True)
    return files[-1]

def as_repo_path(path_text):
    path = Path(path_text)
    return path if path.is_absolute() else REPO / path

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

with tqdm(total=4, desc="Inspect VisA artifacts", unit="step") as pbar:
    manifest_path = latest_file("results/MetaNATH_ViSA_Workflow_*/manifest.json")
    pbar.update(1)
    manifest = read_json(manifest_path)
    pbar.update(1)
    print("manifest:", manifest_path.relative_to(REPO))
    print(json.dumps(manifest, indent=2)[:5000])

    for tier_name, tier in manifest.get("tiers", {}).items():
        report_text = tier.get("acceptance_report")
        if not report_text:
            continue
        report_path = as_repo_path(report_text)
        if not report_path.exists():
            continue
        report = read_json(report_path)
        profile = tier.get("profile", tier_name)
        print(f"\n{tier_name} ({profile}) acceptance:", report_path.relative_to(REPO))
        print("decision:", report.get("decision"), "accepted:", report.get("accepted"))
        print(json.dumps(report.get("checks", []), indent=2))
        phase3_dir = tier.get("phase3_dir")
        if phase3_dir:
            summary_path = as_repo_path(phase3_dir) / "phase3_summary.json"
            if summary_path.exists():
                summary = read_json(summary_path)
                print(f"{tier_name} ({profile}) phase3 summary:", summary_path.relative_to(REPO))
                print(json.dumps(summary, indent=2)[:3000])
    pbar.update(2)


## Package Artifacts

The zip includes JSON/YAML/log/text/markdown artifacts. Checkpoints are excluded by default to keep the output small.

In [ ]:
artifact_zip = Path("/kaggle/working/metanath_visa_kaggle_artifacts.zip")
allowed_suffixes = {".json", ".yaml", ".yml", ".log", ".txt", ".md", ".csv"}
if INCLUDE_CHECKPOINTS_IN_ZIP:
    allowed_suffixes.update({".pt", ".pth", ".ckpt"})

candidate_files = []
for root_name in tqdm(["results", "logs"], desc="scan artifact roots", unit="root"):
    root = REPO / root_name
    if not root.exists():
        continue
    for path in tqdm(root.rglob("*"), desc=f"scan {root_name}", unit="file"):
        if path.is_file() and path.suffix.lower() in allowed_suffixes:
            candidate_files.append(path)

with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in tqdm(candidate_files, desc="zip artifacts", unit="file"):
        zf.write(path, path.relative_to(REPO))

print("artifact_zip:", artifact_zip)
print("files:", len(candidate_files))
print("size_mb:", round(artifact_zip.stat().st_size / (1024 * 1024), 2))
